- Feature Engineering für die Modellierung pro Familiengruppe

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [31]:
# Daten einlesen
BASE_DIR = Path(r"C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales")
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Rohdaten laden

train = pd.read_csv(
    PROCESSED_DIR / "train_family_groups_holidays.csv",
    parse_dates=["date"],
    low_memory=False,
)
test         = pd.read_csv(PROCESSED_DIR / "test_family_groups_holidays.csv",             parse_dates=["date"])
stores       = pd.read_csv(RAW_DIR / "stores.csv")
oil          = pd.read_csv(RAW_DIR / "oil.csv",              parse_dates=["date"])
holidays     = pd.read_csv(RAW_DIR / "holidays_events.csv",  parse_dates=["date"])
transactions = pd.read_csv(RAW_DIR / "transactions.csv",     parse_dates=["date"])


In [32]:
#  Kalender- und Basis-Features ergänzen

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dayofweek"] = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["dayofmonth"] = df["date"].dt.day
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    return df

train = add_calendar_features(train)
test  = add_calendar_features(test)


In [33]:
# Zeitfeatures wie im ersten Versuch

def add_time_features(df):
    df = df.copy()
    df["year"]       = df["date"].dt.year
    df["month"]      = df["date"].dt.month
    df["day"]        = df["date"].dt.day
    df["dayofweek"]  = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["dayofyear"]  = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["quarter"]    = df["date"].dt.quarter

    # Zyklische Encodings
    df["month_sin"]  = np.sin(2 * np.pi * df["month"]     / 12)
    df["month_cos"]  = np.cos(2 * np.pi * df["month"]     / 12)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["woy_sin"]    = np.sin(2 * np.pi * df["weekofyear"]/ 52)
    df["woy_cos"]    = np.cos(2 * np.pi * df["weekofyear"]/ 52)
    df["doy_sin"]    = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * df["dayofyear"] / 365)
    return df

train = add_time_features(train)
test  = add_time_features(test)

# ─────────────────────────────────────────────────────────────────────────────
#    Payday-Features
#    Gehälter im öffentlichen Sektor: 15. und letzter Tag des Monats
# ─────────────────────────────────────────────────────────────────────────────
def add_payday_features(df):
    df = df.copy()
    day      = df["date"].dt.day
    last_day = (df["date"] + pd.offsets.MonthEnd(0)).dt.day

    df["is_payday_15th"]  = (day == 15).astype(int)
    df["is_payday_last"]  = (day == last_day).astype(int)
    df["is_payday"]       = ((df["is_payday_15th"] == 1) | (df["is_payday_last"] == 1)).astype(int)

    # Tage bis zum nächsten Zahltag
    days_until_15   = (15 - day).clip(lower=0)
    days_until_last = (last_day - day).clip(lower=0)
    df["days_until_payday"] = np.minimum(days_until_15, days_until_last)

    # Tage seit dem letzten Zahltag
    days_since_15 = (day - 15).clip(lower=0)
    # Vor dem 15.: Abstand zum letzten Tag des Vormonats
    df["days_since_payday"] = np.where(day >= 15, days_since_15, day - 1)
    # Sonderfall: nach letztem Tag → 0
    df["days_since_payday"] = np.where(
        df["is_payday_last"] == 1, 0, df["days_since_payday"]
    )

    # Kaufsog-Fenster: ±3 Tage um Zahltag
    df["is_payday_window"] = (
        (df["days_until_payday"] <= 3) | (df["days_since_payday"] <= 3)
    ).astype(int)

    return df

train = add_payday_features(train)
test  = add_payday_features(test)



In [34]:
# Sanity-Check
print("Zahltage im Train-Set:", train["is_payday"].sum() // train["store_nbr"].nunique() // train["family"].nunique())
print(train[["date","is_payday","days_since_payday","days_until_payday","is_payday_window"]]
      .drop_duplicates("date").query("is_payday==1").head(6))

Zahltage im Train-Set: 111
             date  is_payday  days_since_payday  days_until_payday  \
24948  2013-01-15          1                  0                  0   
53460  2013-01-31          1                  0                  0   
80190  2013-02-15          1                  0                  0   
103356 2013-02-28          1                  0                  0   
130086 2013-03-15          1                  0                  0   
158598 2013-03-31          1                  0                  0   

        is_payday_window  
24948                  1  
53460                  1  
80190                  1  
103356                 1  
130086                 1  
158598                 1  


In [35]:
# Erdbeben-Features

# Erdbeben-Features
# Magnitude 7.8 Erdbeben am 16. April 2016 in Ecuador.
# Bevölkerung kaufte/spendete Wasser & Grundbedarfsgüter
# → starker Einfluss auf Supermarkt-Umsätze für mehrere Wochen

EARTHQUAKE_DATE = pd.Timestamp("2016-04-16")

def add_earthquake_features(df, eq_date=EARTHQUAKE_DATE, impact_weeks=8):
    df = df.copy()

    delta_days = (df["date"] - eq_date).dt.days

    # Tag des Erdbebens
    df["is_earthquake_day"] = (df["date"] == eq_date).astype(int)

    # Tage seit Erdbeben (negativ = vor Erdbeben → wird als 0 behandelt)
    df["days_since_earthquake"] = delta_days.clip(lower=0)

    # Ob wir im Nachwirkungszeitraum sind (bis impact_weeks Wochen danach)
    max_days = impact_weeks * 7
    in_impact = (delta_days >= 0) & (delta_days <= max_days)

    # Exponentiell abklingendes Signal (Halbwertszeit ~14 Tage)
    df["earthquake_impact"] = np.where(
        in_impact,
        np.exp(-delta_days.clip(lower=0) / 14.0),
        0.0
    )

    # Binäre Flags pro Woche nach dem Erdbeben (Woche 1–4)
    for w in range(1, 5):
        lo = (w - 1) * 7
        hi = w * 7
        df[f"is_post_eq_week{w}"] = (
            (delta_days >= lo) & (delta_days < hi)
        ).astype(int)

    # Wochen 5–8 zusammengefasst als "spätere Nachwirkung"
    df["is_post_eq_week5_8"] = (
        (delta_days >= 28) & (delta_days < 56)
    ).astype(int)

    return df

train = add_earthquake_features(train)
test  = add_earthquake_features(test)

# Sanity-Check: zeige die Woche direkt nach dem Erdbeben
print("\nErdbeben-Features (direkt nach 16.04.2016):")
eq_check = (train[["date","is_earthquake_day","days_since_earthquake",
                    "earthquake_impact","is_post_eq_week1","is_post_eq_week2"]]
            .drop_duplicates("date")
            .query("date >= '2016-04-14' and date <= '2016-05-05'"))
print(eq_check)



Erdbeben-Features (direkt nach 16.04.2016):
              date  is_earthquake_day  days_since_earthquake  \
2131272 2016-04-14                  0                      0   
2133054 2016-04-15                  0                      0   
2134836 2016-04-16                  1                      0   
2136618 2016-04-17                  0                      1   
2138400 2016-04-18                  0                      2   
2140182 2016-04-19                  0                      3   
2141964 2016-04-20                  0                      4   
2143746 2016-04-21                  0                      5   
2145528 2016-04-22                  0                      6   
2147310 2016-04-23                  0                      7   
2149092 2016-04-24                  0                      8   
2150874 2016-04-25                  0                      9   
2152656 2016-04-26                  0                     10   
2154438 2016-04-27                  0                     1

In [36]:
# Ölpreis-Feature, interpoliert, kein Lookahead
oil = oil.rename(columns={"dcoilwtico": "oil_price"})
oil = (
    oil.set_index("date")
    .reindex(pd.date_range(oil["date"].min(), oil["date"].max(), freq="D"))
    .rename_axis("date")
    .reset_index()
)
oil["oil_price"] = oil["oil_price"].interpolate(method="linear")
# Öl-Lags (16, 21, 28 Tage) – kein Leakage im Test
for lag in [16, 21, 28]:
    oil[f"oil_lag_{lag}"] = oil["oil_price"].shift(lag)
# Rolling Mean des Ölpreises
oil["oil_rmean_28"] = oil["oil_price"].rolling(28, min_periods=1).mean()

train = train.merge(oil.drop(columns=["oil_price"]), on="date", how="left")
test  = test.merge(oil.drop(columns=["oil_price"]),  on="date", how="left")
# Wichtig: oil_price selbst weglassen (hätte Lookahead-Problem),
# stattdessen nur die Lags verwenden


In [37]:
# Transactions - Feature (nur Train – im Test unbekannt, daher Lags davon)

train = train.merge(transactions, on=["date", "store_nbr"], how="left")
test["transactions"] = np.nan  # Platzhalter, wird durch Lags ersetzt

In [38]:
# Speichere Basis-Datensatz vor Lags

train.to_csv(PROCESSED_DIR / "train_model_base_v3.csv", index=False)
test.to_csv( PROCESSED_DIR / "test_model_base_v3.csv",  index=False)
print("\nGespeichert: train_model_base_v3.csv, test_model_base_v3.csv")
print("Train shape:", train.shape)
print("Test shape: ", test.shape)


Gespeichert: train_model_base_v3.csv, test_model_base_v3.csv
Train shape: (3000888, 51)
Test shape:  (28512, 49)


In [39]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 51 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id                     int64         
 1   date                   datetime64[ns]
 2   store_nbr              int64         
 3   family                 object        
 4   sales                  float64       
 5   onpromotion            int64         
 6   is_holiday_observed    float64       
 7   is_additional_holiday  float64       
 8   is_bridge_day          float64       
 9   is_work_day_comp       float64       
 10  is_transferred_flag    object        
 11  holiday_names          object        
 12  is_holiday_weekend     float64       
 13  is_any_holiday         int64         
 14  family_group           object        
 15  dayofweek              int32         
 16  weekofyear             int64         
 17  month                  int32         
 18  year                  

In [40]:
train.tail()

,id,date,store_nbr,family,sales,onpromotion,is_holiday_observed,is_additional_holiday,is_bridge_day,is_work_day_comp,...,is_post_eq_week1,is_post_eq_week2,is_post_eq_week3,is_post_eq_week4,is_post_eq_week5_8,oil_lag_16,oil_lag_21,oil_lag_28,oil_rmean_28,transactions
3000883,3000883,2017-08-15,9,POULTRY,438.133,0,1.0,0.0,0.0,0.0,...,0,0,0,0,0,50.046667,47.77,46.4,48.440714,2155.0
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1,1.0,0.0,0.0,0.0,...,0,0,0,0,0,50.046667,47.77,46.4,48.440714,2155.0
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148,1.0,0.0,0.0,0.0,...,0,0,0,0,0,50.046667,47.77,46.4,48.440714,2155.0
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8,1.0,0.0,0.0,0.0,...,0,0,0,0,0,50.046667,47.77,46.4,48.440714,2155.0
3000887,3000887,2017-08-15,9,SEAFOOD,16.000,0,1.0,0.0,0.0,0.0,...,0,0,0,0,0,50.046667,47.77,46.4,48.440714,2155.0


In [41]:
# Schlanke, gruppenweise Lag-/Rolling-Funktion
def add_lag_roll_features_group(df_group: pd.DataFrame) -> pd.DataFrame:
    df_group = df_group.sort_values(["store_nbr", "family", "date"]).copy()

    # Schlanke Lag-Liste
    lag_list = [1, 7, 14, 28, 35, 42, 56, 90, 364, 365, 371]
    for lag in lag_list:
        df_group[f"lag_{lag}"] = (
            df_group
            .groupby(["store_nbr", "family"])["sales"]
            .shift(lag)
        )

    # Rolling Means/Std auf sales
    windows = [7, 14, 28, 56, 90]
    grp = df_group.groupby(["store_nbr", "family"])["sales"]
    for w in windows:
        df_group[f"roll_mean_{w}"] = grp.shift(1).rolling(w).mean()
        df_group[f"roll_std_{w}"]  = grp.shift(1).rolling(w).std()

    # Promotion-Rollings
    if "onpromotion" in df_group.columns:
        grp_promo = df_group.groupby(["store_nbr", "family"])["onpromotion"]
        for w in [7, 28]:
            df_group[f"promo_roll_sum_{w}"] = grp_promo.shift(1).rolling(w).sum()

    return df_group



In [42]:
# 9. Promotions Rolling-Features
#    Wie oft war das Produkt in den letzten N Tagen in Aktion?
#    (nur Lags >= 16, damit kein Leakage im Test)
# 
def add_promo_lag_roll_group(full_g: pd.DataFrame) -> pd.DataFrame:
    """
    Erwartet: full_g mit Spalten:
      - store_nbr
      - family
      - date (sortierbar)
      - onpromotion
    Gibt full_g mit promo_lag_* und promo_rsum_lag16_win* zurück.
    """
    full_g = full_g.sort_values(["store_nbr", "family", "date"]).copy()
    grp = ["store_nbr", "family"]

    # 1) Promo-Lags (z.B. Lags >= 16, damit wir klar nach hinten schauen)
    for lag in [16, 21, 28]:
        full_g[f"promo_lag_{lag}"] = (
            full_g
            .groupby(grp)["onpromotion"]
            .shift(lag)
        )

    # 2) Promotions Rolling-Summen basierend auf Lag-16
    for win in [7, 14, 28]:
        full_g[f"promo_rsum_lag16_win{win}"] = (
            full_g
            .groupby(grp)["promo_lag_16"]
            .transform(lambda s: s.rolling(win, min_periods=1).sum())
        )

    return full_g


In [43]:
# Promotions-Interaktion: Wie reagiert diese Serie auf Promotions?

# Historischer Promotion-Umsatz-Effekt pro Store x Family
def add_promo_lift_group(full_g: pd.DataFrame) -> pd.DataFrame:
    """
    Erwartet: full_g enthält Spalten:
      - store_nbr
      - family
      - sales (NaN im Testteil)
      - onpromotion
    Gibt full_g mit Spalte 'promo_lift_ratio' zurück.
    """
    full_g = full_g.copy()
    grp = ["store_nbr", "family"]

    train_mask = full_g["sales"].notna()

    # Nur Train-Zeilen für das Encoding verwenden
    df_train = full_g.loc[train_mask, ["store_nbr", "family", "sales", "onpromotion"]].copy()

    promo_effect = (
        df_train[df_train["onpromotion"] > 0]
        .groupby(grp)["sales"]
        .mean()
        .rename("mean_sales_on_promo")
        .reset_index()
    )

    no_promo_effect = (
        df_train[df_train["onpromotion"] == 0]
        .groupby(grp)["sales"]
        .mean()
        .rename("mean_sales_no_promo")
        .reset_index()
    )

    promo_lift = promo_effect.merge(no_promo_effect, on=grp, how="left")
    promo_lift["promo_lift_ratio"] = (
        promo_lift["mean_sales_on_promo"] / (promo_lift["mean_sales_no_promo"] + 1e-6)
    )

    full_g = full_g.merge(promo_lift[grp + ["promo_lift_ratio"]], on=grp, how="left")

    return full_g


In [46]:
def add_momentum_features(full_g: pd.DataFrame) -> pd.DataFrame:
    """
    Erwartet: full_g mit Spalten:
      - store_nbr
      - family
      - date
      - sales
    Gibt full_g mit sales_momentum und sales_yoy_ratio zurück.
    """
    full_g = full_g.sort_values(["store_nbr", "family", "date"]).copy()
    grp = ["store_nbr", "family"]

    # 1) Basis-Lags für Momentum/YoY
    # Lag 16 und Lag 364 (falls nicht genug Historie, entstehen NaNs am Anfang)
    full_g["sales_lag16"]  = full_g.groupby(grp)["sales"].shift(16)
    full_g["sales_lag364"] = full_g.groupby(grp)["sales"].shift(364) # Wurde schon früher berechnet

    # 2) Rolling Means auf den gelagten Serien
    # lag16: Fenster 7, 28, 90
    for win in [7, 28, 90]:
        full_g[f"sales_rmean_lag16_win{win}"] = (
            full_g
            .groupby(grp)["sales_lag16"]
            .transform(lambda s: s.rolling(win, min_periods=1).mean())
        )

    # lag364: Fenster 28
    full_g["sales_rmean_lag364_win28"] = (
        full_g
        .groupby(grp)["sales_lag364"]
        .transform(lambda s: s.rolling(28, min_periods=1).mean())
    )

    # 3) Momentum- und YoY-Features
    full_g["sales_momentum"] = (
        full_g["sales_rmean_lag16_win7"] / (full_g["sales_rmean_lag16_win90"] + 1e-6)
    )

    full_g["sales_yoy_ratio"] = (
        full_g["sales_rmean_lag16_win28"] / (full_g["sales_rmean_lag364_win28"] + 1e-6)
    )

    return full_g


In [47]:
# Anwendung pro family_group auf den Kombinierten Datensatz aud train und test, inkl. Target Encoding nur aus Train

train_fe_list = []
test_fe_list  = []

for g in train["family_group"].unique():
    print(f"Verarbeite Gruppe: {g}")
    train_g = train[train["family_group"] == g].copy()
    test_g  = test[test["family_group"] == g].copy()

    full_g = pd.concat([train_g, test_g], ignore_index=True)
    full_g = full_g.sort_values(["store_nbr", "family", "date"])

    # --- Target Encoding NUR aus Train der Gruppe ---
    train_mask_g = full_g["sales"].notna()
    train_g_enc = full_g.loc[train_mask_g, ["store_nbr", "family", "sales"]].copy()
    grp_cols = ["store_nbr", "family"]

    store_family_enc_g = (
        train_g_enc
        .groupby(grp_cols)["sales"]
        .agg(store_family_mean_sales="mean")
        .reset_index()
    )

    store_enc_g = (
        train_g_enc
        .groupby("store_nbr")["sales"]
        .agg(store_mean_sales="mean")
        .reset_index()
    )

    family_enc_g = (
        train_g_enc
        .groupby("family")["sales"]
        .agg(family_mean_sales="mean")
        .reset_index()
    )

    global_mean_g = train_g_enc["sales"].mean()

    # Encodings auf full_g mergen
    full_g = full_g.merge(store_family_enc_g, on=grp_cols,  how="left")
    full_g = full_g.merge(store_enc_g,       on="store_nbr", how="left")
    full_g = full_g.merge(family_enc_g,      on="family",    how="left")

    full_g["store_family_vs_global"] = full_g["store_family_mean_sales"] / (global_mean_g + 1e-6)
    full_g["store_vs_global"]        = full_g["store_mean_sales"]        / (global_mean_g + 1e-6)
    full_g["family_vs_global"]       = full_g["family_mean_sales"]       / (global_mean_g + 1e-6)

    full_g = full_g.drop(columns=["store_family_mean_sales",
                                  "store_mean_sales",
                                  "family_mean_sales"])

   
    # 1) Promo-Lift (historischer Effekt)
    full_g = add_promo_lift_group(full_g)

    # 2) Promo-Lag-/Rolling-Features
    full_g = add_promo_lag_roll_group(full_g)

    # 3) Zeitabhängige Sales-Features (Lags/Rollings/Momentum)
    full_g = add_lag_roll_features_group(full_g)
    full_g = add_momentum_features(full_g)


    # Zurücksplitten in Train/Test für diese Gruppe
    train_g_fe = full_g[full_g["sales"].notna()].copy()
    test_g_fe  = full_g[full_g["sales"].isna()].copy()

    train_fe_list.append(train_g_fe)
    test_fe_list.append(test_g_fe)

train_fe = pd.concat(train_fe_list, ignore_index=True)
test_fe  = pd.concat(test_fe_list,  ignore_index=True)



Verarbeite Gruppe: low_volume_special
Verarbeite Gruppe: mid_volume_mixed
Verarbeite Gruppe: top_volume_food
Verarbeite Gruppe: high_volume_food
Verarbeite Gruppe: low_volume_mixed
Verarbeite Gruppe: grocery_I
Verarbeite Gruppe: low_volume_food


In [48]:
train_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 92 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   id                        int64         
 1   date                      datetime64[ns]
 2   store_nbr                 int64         
 3   family                    object        
 4   sales                     float64       
 5   onpromotion               int64         
 6   is_holiday_observed       float64       
 7   is_additional_holiday     float64       
 8   is_bridge_day             float64       
 9   is_work_day_comp          float64       
 10  is_transferred_flag       object        
 11  holiday_names             object        
 12  is_holiday_weekend        float64       
 13  is_any_holiday            float64       
 14  family_group              object        
 15  dayofweek                 int32         
 16  weekofyear                int64         
 17  month   

In [56]:
# family & family_group als category
for c in ["family", "family_group"]:
    if c in train_fe.columns:
        train_fe[c] = train_fe[c].astype("category")
    
# is_transferred_flag numerisch machen
if "is_transferred_flag" in train_fe.columns:
    train_fe["is_transferred_flag"] = (
        train_fe["is_transferred_flag"]
        .replace({"True": 1, "False": 0})   # falls Strings
        .astype(float)
    )

In [120]:
train_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 92 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   id                        int64         
 1   date                      datetime64[ns]
 2   store_nbr                 category      
 3   family                    category      
 4   sales                     float64       
 5   onpromotion               int64         
 6   is_holiday_observed       float64       
 7   is_additional_holiday     float64       
 8   is_bridge_day             float64       
 9   is_work_day_comp          float64       
 10  is_transferred_flag       float64       
 11  holiday_names             object        
 12  is_holiday_weekend        float64       
 13  is_any_holiday            float64       
 14  family_group              category      
 15  dayofweek                 int32         
 16  weekofyear                int64         
 17  month   

In [57]:
## !!! Diese Liste anpassen!!!

feature_cols = [
    # Basis
    "store_nbr",
    "family",
    "family_group",
    "onpromotion",

    # Kalender (diskret)
    "dayofweek",
    "weekofyear",
    "month",
    "year",
    "dayofmonth",
    "day",
    "dayofyear",
    "quarter",
    "is_weekend",

    # Kalender (Fourier / trigonometrisch)
    "month_sin", "month_cos",
    "dow_sin", "dow_cos",
    "woy_sin", "woy_cos",
    "doy_sin", "doy_cos",

    # Payday-Features
    "is_payday_15th",
    "is_payday_last",
    "is_payday",
    "days_until_payday",
    "days_since_payday",
    "is_payday_window",

    # Earthquake-Features
    "is_earthquake_day",
    "days_since_earthquake",
    "earthquake_impact",
    "is_post_eq_week1",
    "is_post_eq_week2",
    "is_post_eq_week3",
    "is_post_eq_week4",
    "is_post_eq_week5_8",

    # Holiday-Features 
    "is_holiday_observed",
    "is_additional_holiday",
    "is_bridge_day",
    "is_work_day_comp",
    "is_transferred_flag",
    "is_holiday_weekend",
    "is_any_holiday",

    # Oil features
    "oil_lag_16",
    "oil_lag_21",
    "oil_lag_28",
    "oil_rmean_28",
    
    # Lags
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_35",
    "lag_42",
    "lag_56",
    "lag_90",
    "lag_364",   
    "lag_365",
    "lag_371",

    # Rolling-Stats auf Sales
    "roll_mean_7",
    "roll_std_7",
    "roll_mean_14",
    "roll_std_14",
    "roll_mean_28",
    "roll_std_28",
    "roll_mean_56",
    "roll_std_56",
    "roll_mean_90",
    "roll_std_90",

    # Promo-Lags und -Rollings
    "promo_lag_16",
    "promo_lag_21",
    "promo_lag_28",
    "promo_rsum_lag16_win7",
    "promo_rsum_lag16_win14",
    "promo_rsum_lag16_win28",
    "promo_roll_sum_7",
    "promo_roll_sum_28",

    # Transactions 
    "transactions",

    # Momentum/YoY
    "sales_momentum",
    "sales_yoy_ratio",

    # Target-Encodings
    "store_family_vs_global",
    "store_vs_global",
    "family_vs_global",
    "promo_lift_ratio",
]

# Nur Spalten behalten, die wirklich existieren
feature_cols = [c for c in feature_cols if c in train_fe.columns]


# Nur Spalten behalten, die tatsächlich existieren
feature_cols = [c for c in feature_cols if c in train_fe.columns]


In [62]:
# Aufbau erstes Modell mit LightGBM und MLFlow

import lightgbm as lgb
import numpy as np
import mlflow
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import mean_squared_error

In [59]:
# Helper RMLSE
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(np.clip(y_true, 0, None),
                                          np.clip(y_pred, 0, None)))


In [60]:
# MAPE
def mape(y_true, y_pred):
    y_true_c = np.where(y_true == 0, 1e-6, y_true)
    return np.mean(np.abs(y_true - y_pred) / y_true_c) * 100


In [63]:
# Loop für die Schätzung

TARGET = "sales"

# Optional: Zeit-basierter Split-Schlüssel (z.B. letztes Jahr als Valid)
# oder TimeSeriesSplit innerhalb jeder Gruppe

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("store-sales-sarima")

for g, df_g in train_fe.groupby("family_group"):
    print(f"Trainiere Gruppe: {g}, n={len(df_g)}")

    df_g = df_g.sort_values("date").copy()

    # Zeilen mit vollständigen Lags (z.B. lag_28) behalten
    df_g = df_g.dropna(subset=[c for c in ["lag_28"] if c in df_g.columns])

    cat_cols = ["family", "family_group"]

       
    X = df_g[feature_cols]
    y = df_g[TARGET]

    # Log-Target
    y_log = np.log1p(np.clip(y, 0, None))

    # Kategoriale Features für LightGBM
    cat_features = [c for c in ["store_nbr", "family", "family_group"] if c in feature_cols]

    # Zeitbasierter Split (einfacher 80/20-Date-Split)
    split_date = df_g["date"].quantile(0.8)
    train_idx = df_g["date"] <= split_date
    valid_idx = df_g["date"] > split_date

    X_train, y_train = X[train_idx],  y_log[train_idx]
    X_valid, y_valid = X[valid_idx],  y_log[valid_idx]

    train_set = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_features or None
    )
    valid_set = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=cat_features or None
    )

    params = {
        "objective": "regression",   # log1p-Target
        "metric": "rmse",            # RMSE im Log-Raum
        "learning_rate": 0.05,
        "num_leaves": 64,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "seed": 42,
        "verbosity": -1,
    }

    with mlflow.start_run(run_name=f"lgbm_{g}"):
        mlflow.log_params({"group": g, **params})

        callbacks = [
            lgb.early_stopping(stopping_rounds=200, verbose=True),
            lgb.log_evaluation(period=200),
        ]
                
        model = lgb.train(
            params,
            train_set,
            num_boost_round=3000,
            valid_sets=[train_set, valid_set],
            valid_names=["train", "valid"],
            callbacks = callbacks, 
        )

        # Vorhersage im Log-Raum
        y_pred_log_valid = model.predict(
            X_valid,
            num_iteration=model.best_iteration
        )

        # RMSE im Log-Raum (Diagnose)
        rmse_log = np.sqrt(
            np.mean((y_valid - y_pred_log_valid) ** 2)
        )

        # Zurück in den Sales-Raum
        y_valid_orig = np.expm1(y_valid)
        y_pred_valid = np.expm1(y_pred_log_valid)

        # Metriken im Sales-Raum
        score_rmsle = rmsle(y_valid_orig, y_pred_valid)
        score_mape  = mape(y_valid_orig, y_pred_valid)
        rmse_val    = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid))

        mlflow.log_metric("rmse_log_valid", rmse_log)
        mlflow.log_metric("rmsle_valid",   score_rmsle)
        mlflow.log_metric("mape_valid",    score_mape)
        mlflow.log_metric("rmse_valid",    rmse_val)
        mlflow.log_metric("best_iteration", model.best_iteration)

        # Feature Importance
        fi = model.feature_importance(importance_type="gain")
        fi_df = (
            pd.DataFrame({"feature": model.feature_name(), "importance": fi})
            .sort_values("importance", ascending=False)
        )
        Path("models").mkdir(exist_ok=True)
        fi_path = f"feature_importance_{g}.csv"
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        # Modell speichern
        model_path = f"models/lgbm_{g}.txt"
        model.save_model(model_path)
        mlflow.log_artifact(model_path)


C:\Users\nelid\AppData\Local\Temp\ipykernel_33240\3163883399.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, df_g in train_fe.groupby("family_group"):


Trainiere Gruppe: grocery_I, n=90936
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.111052	valid's rmse: 0.165583
Early stopping, best iteration is:
[135]	train's rmse: 0.120414	valid's rmse: 0.162247
🏃 View run lgbm_grocery_I at: http://127.0.0.1:5000/#/experiments/1/runs/904f5423179c48b19303ffeaa3ade1f5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere Gruppe: high_volume_food, n=545616
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.177379	valid's rmse: 0.213839
[400]	train's rmse: 0.165782	valid's rmse: 0.210119
[600]	train's rmse: 0.15828	valid's rmse: 0.208662
[800]	train's rmse: 0.15238	valid's rmse: 0.20794
[1000]	train's rmse: 0.147457	valid's rmse: 0.207456
[1200]	train's rmse: 0.143134	valid's rmse: 0.207187
[1400]	train's rmse: 0.139301	valid's rmse: 0.207179
Early stopping, best iteration is:
[1312]	train's rmse: 0.140936	valid's rmse: 0.20705
🏃 View run lgbm_high_volume_food at: h

In [66]:
# XGB Loop:

# from pathlib import Path

# import numpy as np
# import pandas as pd
from xgboost import XGBRegressor
# from sklearn.metrics import mean_squared_error
# import mlflow


# -----------------------------
# Metriken
# -----------------------------
def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))

def mape(y_true, y_pred):
    y_true_c = np.where(y_true == 0, 1e-6, y_true)
    return np.mean(np.abs(y_true - y_pred) / y_true_c) * 100


TARGET = "sales"

# WICHTIG: sicherstellen, dass Kategorien korrekt gecastet sind
for c in ["family", "family_group"]:
    if c in train_fe.columns:
        train_fe[c] = train_fe[c].astype("category")

if "is_transferred_flag" in train_fe.columns:
    train_fe["is_transferred_flag"] = (
        train_fe["is_transferred_flag"]
        .replace({"True": 1, "False": 0})
        .astype(float)
    )

# mlflow.set_experiment("store-sales-family-groups-xgb")


# -----------------------------
# Loop pro family_group
# -----------------------------
for g, df_g in train_fe.groupby("family_group"):
    print(f"Trainiere XGB für Gruppe: {g}, n={len(df_g)}")

    df_g = df_g.sort_values("date").copy()

    # Zeilen mit vollständigen Lags (z.B. lag_28) behalten
    df_g = df_g.dropna(subset=[c for c in ["lag_28"] if c in df_g.columns])

    X = df_g[feature_cols].copy()
    y = df_g[TARGET].copy()

    # Log-Target
    y_log = np.log1p(np.clip(y, 0, None))

    # Kategoriale Features: für XGB müssen sie numerisch/encoded sein.
    # Pandas-Categories -> Codes
    for c in ["family", "family_group"]:
        if c in X.columns and str(X[c].dtype) == "category":
            X[c] = X[c].cat.codes.astype("int32")

    # Zeitbasierter Split (einfacher 80/20-Date-Split)
    split_date = df_g["date"].quantile(0.8)
    train_idx = df_g["date"] <= split_date
    valid_idx = df_g["date"] > split_date

    X_train, y_train = X[train_idx], y_log[train_idx]
    X_valid, y_valid = X[valid_idx], y_log[valid_idx]

    # XGBoost-Modell
    params = {
        "tree_method": "hist",          # "gpu_hist", falls GPU verfügbar
        "max_depth": 8,
        "learning_rate": 0.05,
        "n_estimators": 3000,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "reg:squarederror",
        "random_state": 42,
     }
    
    with mlflow.start_run(run_name=f"xgb_{g}"):
        mlflow.log_params({"group": g, **params})
    
        model = XGBRegressor(**params)
        model.fit(
            X_train,
            y_train,
        )

        # Vorhersage im Log-Raum (best_iteration nutzt XGB intern)
        y_pred_log_valid = model.predict(X_valid)

        # RMSE im Log-Raum (Diagnose)
        rmse_log = np.sqrt(np.mean((y_valid - y_pred_log_valid) ** 2))

        # Zurück in den Sales-Raum
        y_valid_orig = np.expm1(y_valid)
        y_pred_valid = np.expm1(y_pred_log_valid)

        # Metriken im Sales-Raum
        score_rmsle = rmsle(y_valid_orig, y_pred_valid)
        score_mape  = mape(y_valid_orig, y_pred_valid)
        rmse_val    = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid))

        mlflow.log_metric("rmse_log_valid", rmse_log)
        mlflow.log_metric("rmsle_valid",   score_rmsle)
        mlflow.log_metric("mape_valid",    score_mape)
        mlflow.log_metric("rmse_valid",    rmse_val)

        # Feature Importances speichern (Gain-basiert)
        fi = model.feature_importances_
        fi_df = (
            pd.DataFrame({"feature": feature_cols, "importance": fi})
            .sort_values("importance", ascending=False)
        )
        Path("models").mkdir(exist_ok=True)
        fi_path = f"feature_importance_xgb_{g}.csv"
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        # Modell speichern
        model_path = f"models/xgb_{g}.json"
        model.save_model(model_path)
        mlflow.log_artifact(model_path)

        print(
            f"Gruppe {g}: RMSE_log={rmse_log:.4f}, "
            f"RMSLE={score_rmsle:.4f}, MAPE={score_mape:.2f}%, RMSE={rmse_val:.4f}"
        )


C:\Users\nelid\AppData\Local\Temp\ipykernel_33240\792042841.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, df_g in train_fe.groupby("family_group"):


Trainiere XGB für Gruppe: grocery_I, n=90936
Gruppe grocery_I: RMSE_log=0.1657, RMSLE=0.1633, MAPE=166631.22%, RMSE=967.1051
🏃 View run xgb_grocery_I at: http://127.0.0.1:5000/#/experiments/1/runs/aa683bae3a004a3b91c5a91a9e2c732a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere XGB für Gruppe: high_volume_food, n=545616
Gruppe high_volume_food: RMSE_log=0.2085, RMSLE=0.2085, MAPE=385573.14%, RMSE=167.5969
🏃 View run xgb_high_volume_food at: http://127.0.0.1:5000/#/experiments/1/runs/31611f2374754cf4bdc830cd9476617c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere XGB für Gruppe: low_volume_food, n=272808
Gruppe low_volume_food: RMSE_log=0.3591, RMSLE=0.3590, MAPE=6762086.76%, RMSE=393.3865
🏃 View run xgb_low_volume_food at: http://127.0.0.1:5000/#/experiments/1/runs/0c673105a2ec4a11841f644be984a7ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere XGB für Gruppe: low_volume_mixed, n=272808
Gruppe low_volume_mixed: RMSE_log=0

In [67]:
# Getunetes XGB:

from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import mlflow


# -----------------------------
# Metriken
# -----------------------------
def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))

def mape(y_true, y_pred):
    y_true_c = np.where(y_true == 0, 1e-6, y_true)
    return np.mean(np.abs(y_true - y_pred) / y_true_c) * 100


TARGET = "sales"

# Typen für Kategorien und problematische Spalten in train_fe einmal sauber setzen
for c in ["family", "family_group"]:
    if c in train_fe.columns:
        train_fe[c] = train_fe[c].astype("category")

if "is_transferred_flag" in train_fe.columns:
    train_fe["is_transferred_flag"] = (
        train_fe["is_transferred_flag"]
        .replace({"True": 1, "False": 0})
        .astype(float)
    )

#mlflow.set_experiment("store-sales-family-groups-xgb-tuned")


# -----------------------------
# Loop pro family_group
# -----------------------------
for g, df_g in train_fe.groupby("family_group"):
    print(f"Trainiere getunten XGB für Gruppe: {g}, n={len(df_g)}")

    df_g = df_g.sort_values("date").copy()

    # Zeilen mit vollständigen Lags (z.B. lag_28) behalten
    df_g = df_g.dropna(subset=[c for c in ["lag_28"] if c in df_g.columns])

    X = df_g[feature_cols].copy()
    y = df_g[TARGET].copy()

    # Log-Target
    y_log = np.log1p(np.clip(y, 0, None))

    # Categoricals für XGB in Codes umwandeln
    for c in ["family", "family_group"]:
        if c in X.columns and str(X[c].dtype) == "category":
            X[c] = X[c].cat.codes.astype("int32")

    # Zeitbasierter Split (einfacher 80/20-Date-Split)
    split_date = df_g["date"].quantile(0.8)
    train_idx = df_g["date"] <= split_date
    valid_idx = df_g["date"] > split_date

    X_train, y_train = X[train_idx], y_log[train_idx]
    X_valid, y_valid = X[valid_idx], y_log[valid_idx]

    # Getunte XGB-Parameter (aus globalem Modell)
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        # eval_metric wird von deiner Version in fit evtl. ignoriert, schadet aber nicht
        "eta": 0.05,          # learning_rate
        "lambda": 1.0,        # L2-Regularisierung
        "alpha": 0.0,         # L1-Regularisierung
        "max_depth": 10,
        "min_child_weight": 3,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    }

    with mlflow.start_run(run_name=f"xgb_tuned_{g}"):
        mlflow.log_params({"group": g, **params})

        model = XGBRegressor(
            objective=params["objective"],
            tree_method=params["tree_method"],
            learning_rate=params["eta"],
            reg_lambda=params["lambda"],
            reg_alpha=params["alpha"],
            max_depth=params["max_depth"],
            min_child_weight=params["min_child_weight"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            n_estimators=1000,   # moderat, da kein Early Stopping
            random_state=42,
        )

        model.fit(
            X_train,
            y_train,
            # keine eval_set / early_stopping, da deine Version die Argumente nicht kennt
        )

        # Vorhersage im Log-Raum
        y_pred_log_valid = model.predict(X_valid)

        # RMSE im Log-Raum (Diagnose)
        rmse_log = np.sqrt(np.mean((y_valid - y_pred_log_valid) ** 2))

        # Zurück in den Sales-Raum
        y_valid_orig = np.expm1(y_valid)
        y_pred_valid = np.expm1(y_pred_log_valid)

        # Metriken im Sales-Raum
        score_rmsle = rmsle(y_valid_orig, y_pred_valid)
        score_mape  = mape(y_valid_orig, y_pred_valid)
        rmse_val    = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid))

        mlflow.log_metric("rmse_log_valid", rmse_log)
        mlflow.log_metric("rmsle_valid",   score_rmsle)
        mlflow.log_metric("mape_valid",    score_mape)
        mlflow.log_metric("rmse_valid",    rmse_val)

        # Feature Importances speichern
        fi = model.feature_importances_
        fi_df = (
            pd.DataFrame({"feature": feature_cols, "importance": fi})
            .sort_values("importance", ascending=False)
        )
        Path("models").mkdir(exist_ok=True)
        fi_path = f"feature_importance_xgb_tuned_{g}.csv"
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        # Modell speichern
        model_path = f"models/xgb_tuned_{g}.json"
        model.save_model(model_path)
        mlflow.log_artifact(model_path)

        print(
            f"Gruppe {g}: RMSE_log={rmse_log:.4f}, "
            f"RMSLE={score_rmsle:.4f}, MAPE={score_mape:.2f}%, RMSE={rmse_val:.4f}"
        )


C:\Users\nelid\AppData\Local\Temp\ipykernel_33240\2915334665.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, df_g in train_fe.groupby("family_group"):


Trainiere getunten XGB für Gruppe: grocery_I, n=90936
Gruppe grocery_I: RMSE_log=0.1654, RMSLE=0.1630, MAPE=126104.42%, RMSE=951.9422
🏃 View run xgb_tuned_grocery_I at: http://127.0.0.1:5000/#/experiments/1/runs/6978b29978cf4a22b75235d7195b1f08
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere getunten XGB für Gruppe: high_volume_food, n=545616
Gruppe high_volume_food: RMSE_log=0.2045, RMSLE=0.2042, MAPE=235661.20%, RMSE=170.4828
🏃 View run xgb_tuned_high_volume_food at: http://127.0.0.1:5000/#/experiments/1/runs/d64a0f5fae47487784125f20c61328d6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere getunten XGB für Gruppe: low_volume_food, n=272808
Gruppe low_volume_food: RMSE_log=0.3590, RMSLE=0.3590, MAPE=11852993.26%, RMSE=393.3533
🏃 View run xgb_tuned_low_volume_food at: http://127.0.0.1:5000/#/experiments/1/runs/bdef54c31b4645339f7c7a5879f74be0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Trainiere getunten XGB für Gruppe: low_vol

In [69]:
# Catboost

from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
import mlflow

def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))

def mape(y_true, y_pred):
    y_true_c = np.where(y_true == 0, 1e-6, y_true)
    return np.mean(np.abs(y_true - y_pred) / y_true_c) * 100

TARGET = "sales"

# Typen in train_fe
for c in ["family", "family_group", "store_nbr"]:
    if c in train_fe.columns:
        train_fe[c] = train_fe[c].astype("category")

if "is_transferred_flag" in train_fe.columns:
    train_fe["is_transferred_flag"] = (
        train_fe["is_transferred_flag"]
        .replace({"True": 1, "False": 0})
        .astype(float)
    )

# mlflow.set_experiment("store-sales-family-groups-catboost")

for g, df_g in train_fe.groupby("family_group"):
    print(f"Trainiere CatBoost für Gruppe: {g}, n={len(df_g)}")

    df_g = df_g.sort_values("date").copy()
    df_g = df_g.dropna(subset=[c for c in ["lag_28"] if c in df_g.columns])

    X = df_g[feature_cols].copy()
    y = df_g[TARGET].copy()
    y_log = np.log1p(np.clip(y, 0, None))

    # Kategoriale Features: Indizes der Spalten in X
    cat_features = [c for c in ["store_nbr", "family", "family_group"] if c in X.columns]
    cat_features_idx = [X.columns.get_loc(c) for c in cat_features]

    # Zeit-Split 80/20
    split_date = df_g["date"].quantile(0.8)
    train_idx = df_g["date"] <= split_date
    valid_idx = df_g["date"] > split_date

    X_train, y_train = X[train_idx], y_log[train_idx]
    X_valid, y_valid = X[valid_idx], y_log[valid_idx]

    train_pool = Pool(X_train, label=y_train, cat_features=cat_features_idx)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features_idx)

    params = {
        "loss_function": "RMSE",   # im Log-Raum
        "depth": 8,
        "learning_rate": 0.05,
        "iterations": 3000,
        "random_seed": 42,
        "l2_leaf_reg": 3.0,
        "bootstrap_type": "Bayesian",
        "od_type": "Iter",
        "od_wait": 200,
        "verbose": 200,
    }

    with mlflow.start_run(run_name=f"cat_{g}"):
        mlflow.log_params({"group": g, **params})

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=valid_pool)

        # Vorhersage im Log-Raum
        y_pred_log_valid = model.predict(valid_pool)

        # RMSE im Log-Raum
        rmse_log = np.sqrt(np.mean((y_valid - y_pred_log_valid) ** 2))

        # Zurück in den Sales-Raum
        y_valid_orig = np.expm1(y_valid)
        y_pred_valid = np.expm1(y_pred_log_valid)

        score_rmsle = rmsle(y_valid_orig, y_pred_valid)
        score_mape  = mape(y_valid_orig, y_pred_valid)
        rmse_val    = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid))

        mlflow.log_metric("rmse_log_valid", rmse_log)
        mlflow.log_metric("rmsle_valid",   score_rmsle)
        mlflow.log_metric("mape_valid",    score_mape)
        mlflow.log_metric("rmse_valid",    rmse_val)

        # Feature Importances (PredictionValuesChange)
        fi = model.get_feature_importance(train_pool, type="PredictionValuesChange")
        fi_df = (
            pd.DataFrame({"feature": X.columns, "importance": fi})
            .sort_values("importance", ascending=False)
        )
        Path("models").mkdir(exist_ok=True)
        fi_path = f"feature_importance_cat_{g}.csv"
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)

        model_path = f"models/cat_{g}.cbm"
        model.save_model(model_path)
        mlflow.log_artifact(model_path)

        print(
            f"Gruppe {g}: RMSE_log={rmse_log:.4f}, "
            f"RMSLE={score_rmsle:.4f}, MAPE={score_mape:.2f}%, RMSE={rmse_val:.4f}"
        )


C:\Users\nelid\AppData\Local\Temp\ipykernel_33240\101097559.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, df_g in train_fe.groupby("family_group"):


Trainiere CatBoost für Gruppe: grocery_I, n=90936
0:	learn: 2.3033952	test: 1.4697552	best: 1.4697552 (0)	total: 274ms	remaining: 13m 41s
200:	learn: 0.1444558	test: 0.2682835	best: 0.2680686 (187)	total: 22.2s	remaining: 5m 9s
400:	learn: 0.1268780	test: 0.2583167	best: 0.2583167 (400)	total: 44.5s	remaining: 4m 48s
600:	learn: 0.1158965	test: 0.2550408	best: 0.2550342 (596)	total: 1m 6s	remaining: 4m 25s
800:	learn: 0.1077015	test: 0.2523409	best: 0.2523409 (800)	total: 1m 27s	remaining: 4m
1000:	learn: 0.1020730	test: 0.2501677	best: 0.2501600 (998)	total: 1m 49s	remaining: 3m 37s
1200:	learn: 0.0975394	test: 0.2506315	best: 0.2499559 (1056)	total: 2m 10s	remaining: 3m 15s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.2499558949
bestIteration = 1056

Shrink model to first 1057 iterations.
Gruppe grocery_I: RMSE_log=0.2500, RMSLE=0.2500, MAPE=10185074.71%, RMSE=993.8673
🏃 View run cat_grocery_I at: http://127.0.0.1:5000/#/experiments/1/runs/c617e6ce6c2d416580ae

In [76]:
### Predictions versuchen zu verbessern über CV

# Generic TimeSeries‑CV‑Wrapper mit OOF‑Preds
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

'''
def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))
'''

def run_timeseries_cv(
    df,
    feature_cols,
    target_col="sales",
    n_splits=4,
    model_builder=None,
    model_name="model",
):
    df = df.sort_values("date").reset_index(drop=True).copy()
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    y_log = np.log1p(np.clip(y, 0, None))

    # Categorical zu Codes für alle Nicht-CatBoost-Modelle
    cat_cols = [c for c in ["store_nbr", "family", "family_group"] if c in X.columns]
    for c in cat_cols:
        if str(X[c].dtype) == "category":
            X[c] = X[c].cat.codes.astype("int32")

    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof_pred_log = np.full(len(df), np.nan, dtype=float)
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(tscv.split(X), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y_log.iloc[train_idx], y_log.iloc[valid_idx]

        model = model_builder()

        if model_name.lower().startswith("cat"):
            from catboost import Pool
            # für CatBoost brauchen wir wieder die Rohdaten inkl. categoricals:
            X_train_raw = df.loc[train_idx, feature_cols].copy()
            X_valid_raw = df.loc[valid_idx, feature_cols].copy()
            cat_features = [c for c in ["store_nbr", "family", "family_group"] if c in X_train_raw.columns]
            cat_idx = [X_train_raw.columns.get_loc(c) for c in cat_features]

            train_pool = Pool(X_train_raw, label=y_train, cat_features=cat_idx)
            valid_pool = Pool(X_valid_raw, label=y_valid, cat_features=cat_idx)
            model.fit(train_pool, eval_set=valid_pool, verbose=False)
            y_pred_log = model.predict(valid_pool)
        else:
            model.fit(X_train, y_train)
            y_pred_log = model.predict(X_valid)

        rmse_log = np.sqrt(mean_squared_error(y_valid, y_pred_log))
        y_valid_orig = np.expm1(y_valid)
        y_pred_orig = np.expm1(y_pred_log)
        score_rmsle = rmsle(y_valid_orig, y_pred_orig)
        rmse_val = np.sqrt(mean_squared_error(y_valid_orig, y_pred_orig))

        oof_pred_log[valid_idx] = y_pred_log

        print(
            f"{model_name} | Fold {fold}: "
            f"RMSE_log={rmse_log:.4f}, RMSLE={score_rmsle:.4f}, RMSE={rmse_val:.4f}"
        )
        fold_scores.append(
            {"fold": fold, "rmse_log": rmse_log, "rmsle": score_rmsle, "rmse": rmse_val}
        )

    return oof_pred_log, fold_scores



In [77]:
#  Drei Builder‑Funktionen für XGB, LGBM, CatBoost

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


def build_xgb():
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        learning_rate=0.05,
        reg_lambda=1.0,
        reg_alpha=0.0,
        max_depth=10,
        min_child_weight=3,
        subsample=0.9,
        colsample_bytree=0.9,
        n_estimators=1000,
        random_state=42,
    )


def build_lgbm():
    return LGBMRegressor(
        objective="regression",
        boosting_type="gbdt",
        learning_rate=0.05,
        num_leaves=255,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.9,
        n_estimators=2000,
        reg_lambda=0.0,
        reg_alpha=0.0,
        random_state=42,
    )


def build_cat():
    return CatBoostRegressor(
        loss_function="RMSE",
        depth=8,
        learning_rate=0.05,
        iterations=3000,
        random_seed=42,
        l2_leaf_reg=3.0,
        bootstrap_type="Bayesian",
        od_type="Iter",
        od_wait=200,
        verbose=False,
    )


In [78]:
#  OOF‑Preds für alle drei Modelle erzeugen

# Annahme: train_fe hat 'date', 'sales' und feature_cols
df_cv = train_fe.sort_values("date").reset_index(drop=True).copy()

oof_xgb_log, scores_xgb = run_timeseries_cv(
    df_cv, feature_cols, target_col="sales", n_splits=4, model_builder=build_xgb, model_name="xgb"
)

oof_lgbm_log, scores_lgbm = run_timeseries_cv(
    df_cv, feature_cols, target_col="sales", n_splits=4, model_builder=build_lgbm, model_name="lgbm"
)

oof_cat_log, scores_cat = run_timeseries_cv(
    df_cv, feature_cols, target_col="sales", n_splits=4, model_builder=build_cat, model_name="catboost"
)


xgb | Fold 1: RMSE_log=0.5891, RMSLE=0.5888, RMSE=296.5716
xgb | Fold 2: RMSE_log=0.4658, RMSLE=0.4655, RMSE=283.9392
xgb | Fold 3: RMSE_log=0.5002, RMSLE=0.5000, RMSE=368.0794
xgb | Fold 4: RMSE_log=0.4007, RMSLE=0.4005, RMSE=279.8252
lgbm | Fold 1: RMSE_log=0.5805, RMSLE=0.5803, RMSE=295.9504
lgbm | Fold 2: RMSE_log=0.4764, RMSLE=0.4760, RMSE=244.1150
lgbm | Fold 3: RMSE_log=0.4934, RMSLE=0.4932, RMSE=368.6900
lgbm | Fold 4: RMSE_log=0.3980, RMSLE=0.3978, RMSE=272.6372
catboost | Fold 1: RMSE_log=0.5828, RMSLE=0.5826, RMSE=381.7241
catboost | Fold 2: RMSE_log=0.4157, RMSLE=0.4155, RMSE=243.0057
catboost | Fold 3: RMSE_log=0.4686, RMSLE=0.4685, RMSE=410.2056
catboost | Fold 4: RMSE_log=0.4211, RMSLE=0.4210, RMSE=326.8902


In [79]:
# Check RMLSE
print("XGB mean RMSLE:", np.mean([s["rmsle"] for s in scores_xgb]))
print("LGBM mean RMSLE:", np.mean([s["rmsle"] for s in scores_lgbm]))
print("CAT mean RMSLE:", np.mean([s["rmsle"] for s in scores_cat]))


XGB mean RMSLE: 0.4886858539490092
LGBM mean RMSLE: 0.4868224570550269
CAT mean RMSLE: 0.47192134907386224


In [98]:
# Einfaches OOF‑Ensemble bauen und bewerten
# sicherstellen, dass alle OOF‑Arrays gefüllt sind
mask_valid = ~np.isnan(oof_xgb_log) & ~np.isnan(oof_lgbm_log) & ~np.isnan(oof_cat_log)

y_log = np.log1p(np.clip(df_cv["sales"].values, 0, None))

# Einfacher Mittelwert im Log‑Raum
oof_ens_log = (
    oof_xgb_log[mask_valid]
    + oof_lgbm_log[mask_valid]
    + oof_cat_log[mask_valid]
) / 3.0

y_valid_log = y_log[mask_valid]

# Metriken im Log‑Raum
rmse_log_ens = np.sqrt(mean_squared_error(y_valid_log, oof_ens_log))

# zurück in Sales‑Raum
y_valid_orig = np.expm1(y_valid_log)
y_pred_orig = np.expm1(oof_ens_log)

def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)      # <‑ wichtig
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))


rmsle_ens = rmsle(y_valid_orig, y_pred_orig)
rmse_ens = np.sqrt(mean_squared_error(y_valid_orig, y_pred_orig))

print(
    f"Ensemble OOF: RMSE_log={rmse_log_ens:.4f}, "
    f"RMSLE={rmsle_ens:.4f}, RMSE={rmse_ens:.4f}"
)



Ensemble OOF: RMSE_log=3.6910, RMSLE=3.6907, RMSE=1596.2171


In [99]:
'''
## Debug
mask_valid = ~np.isnan(oof_xgb_log) & ~np.isnan(oof_lgbm_log) & ~np.isnan(oof_cat_log)

w_x, w_l, w_c = 0.4, 0.4, 0.2
oof_ensw_log = (
    w_x * oof_xgb_log[mask_valid]
    + w_l * oof_lgbm_log[mask_valid]
    + w_c * oof_cat_log[mask_valid]
)

y_true = df_cv["sales"].values[mask_valid]     # ORIGINAL sales
y_pred = np.expm1(oof_ensw_log)                # zurücktransformiert
# kein weiteres Clampen nötig, das macht rmsle intern:
print("RMSLE Ensemble:", rmsle(y_true, y_pred))
'''

RMSLE Ensemble: 3.690241564759821


In [100]:
'''
# Debug
# 1) Wie sieht y_log im CV-Wrapper aus?
df_cv = train_fe.sort_values("date").reset_index(drop=True).copy()
y_raw = df_cv["sales"].values
y_log = np.log1p(np.clip(y_raw, 0, None))

print("y_raw[:5]:", y_raw[:5])
print("y_log[:5]:", y_log[:5])
'''

y_raw[:5]: [0. 0. 0. 0. 0.]
y_log[:5]: [0. 0. 0. 0. 0.]


In [101]:
'''
# Debug
# 2) Einzelmodell-OOF prüfen
y_true = y_raw[mask_valid]                     # ORIGINAL sales
y_pred_xgb = np.expm1(oof_xgb_log[mask_valid]) # XGB-OFF-Preds zurücktransformiert

print("XGB oof_log[:5]:", oof_xgb_log[mask_valid][:5])
print("XGB oof back[:5]:", y_pred_xgb[:5])
print("RMSLE XGB einzeln:", rmsle(y_true, y_pred_xgb))
'''

XGB oof_log[:5]: [ 0.20627281  2.44654918 -0.00319239  3.08535695 -0.00885411]
XGB oof back[:5]: [ 2.29088467e-01  1.05484263e+01 -3.18730238e-03  2.08752738e+01
 -8.81502759e-03]
RMSLE XGB einzeln: 3.6887404185632464


In [91]:
# Alternative: gewichtetes Ensemble machen, Gewichte proportional zu 1/RMSLE der Einzelmodelle oder sie per Hand einstellen.

rmsle_x = np.mean([s["rmsle"] for s in scores_xgb])
rmsle_l = np.mean([s["rmsle"] for s in scores_lgbm])
rmsle_c = np.mean([s["rmsle"] for s in scores_cat])

# 2) Glättung, damit Unterschiede nicht zu krass sind
eps = 1e-3
w_x = 1 / (rmsle_x + eps)
w_l = 1 / (rmsle_l + eps)
w_c = 1 / (rmsle_c + eps)

# Optional: zusätzlich alle Gewichte auf Summe 1 normalisieren
w_sum = w_x + w_l + w_c
w_x /= w_sum
w_l /= w_sum
w_c /= w_sum

oof_ensw_log = (
    w_x * oof_xgb_log[mask_valid]
    + w_l * oof_lgbm_log[mask_valid]
    + w_c * oof_cat_log[mask_valid]
)



In [93]:
np.mean(oof_ensw_log)

np.float64(3.1203562879234648)

In [95]:
import mlflow
from sklearn.metrics import mean_squared_error
import numpy as np

with mlflow.start_run(run_name="ensemble_oof_weighted"):
    # Einzelmodell-Metriken (aus den Fold-Scores, im ORIGINALraum berechnet!)
    mlflow.log_metric("rmsle_xgb", rmsle_x)
    mlflow.log_metric("rmsle_lgbm", rmsle_l)
    mlflow.log_metric("rmsle_cat", rmsle_c)

    # --- 1) Log-Raum nur für RMSE_log verwenden ---
    y_log = np.log1p(np.clip(df_cv["sales"].values, 0, None))   # log1p(sales)
    y_valid_log = y_log[mask_valid]                             # log-Target
    rmse_log_ensw = np.sqrt(mean_squared_error(y_valid_log, oof_ensw_log))
    mlflow.log_metric("rmse_log_ens_weighted", rmse_log_ensw)

    # --- 2) Originalraum für RMSLE / RMSE ---
    y_valid_orig = df_cv["sales"].values[mask_valid]            # ORIGINAL sales
    y_pred_orig = np.expm1(oof_ensw_log)                        # aus log zurück

    rmsle_ensw = rmsle(y_valid_orig, y_pred_orig)
    rmse_ensw = np.sqrt(mean_squared_error(y_valid_orig, y_pred_orig))

    mlflow.log_metric("rmsle_ens_weighted", rmsle_ensw)
    mlflow.log_metric("rmse_ens_weighted", rmse_ensw)



🏃 View run ensemble_oof_weighted at: http://127.0.0.1:5000/#/experiments/1/runs/5c9cd932321a41e6b31442c90045cea7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [96]:
# 1) Maske wie bisher
mask_valid = ~np.isnan(oof_xgb_log) & ~np.isnan(oof_lgbm_log) & ~np.isnan(oof_cat_log)

# 2) Ensemble im LOG-Raum (z.B. mit festen Gewichten)
w_x, w_l, w_c = 0.4, 0.4, 0.2
oof_ensw_log = (
    w_x * oof_xgb_log[mask_valid]
    + w_l * oof_lgbm_log[mask_valid]
    + w_c * oof_cat_log[mask_valid]
)

def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)      # <‑ wichtig
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))


# 3) y_true und y_pred im ORIGINALraum
y_true = df_cv["sales"].values[mask_valid]        # ACHTUNG: NICHT geloggt
y_pred = np.expm1(oof_ensw_log)                   # log-Preds zurücktransformieren

print("min/max y_true:", float(y_true.min()), float(y_true.max()))
print("min/max y_pred:", float(y_pred.min()), float(y_pred.max()))
print("mean y_true:", float(y_true.mean()))
print("mean y_pred:", float(y_pred.mean()))
print("RMSLE Ensemble:", rmsle(y_true, y_pred))


min/max y_true: 0.0 124717.0
min/max y_pred: -0.7812476611376622 18245.529349960278
mean y_true: 394.6754957152123
mean y_pred: 376.66041116418575
RMSLE Ensemble: 3.690241564759821


In [102]:
## Submission vorbereiten

from copy import deepcopy

def prepare_xy(df, feature_cols, target_col="sales"):
    df = df.sort_values("date").copy()
    X = df[feature_cols].copy()
    y = df[target_col].values

    # Categorical → Codes für Booster (nicht für CatBoost)
    cat_cols = [c for c in ["store_nbr", "family", "family_group"] if c in X.columns]
    for c in cat_cols:
        if str(X[c].dtype) == "category":
            X[c] = X[c].cat.codes.astype("int32")

    y_log = np.log1p(np.clip(y, 0, None))
    return X, y, y_log


In [122]:
# Volle Modelle trainieren und Test‑Preds erzeugen

from catboost import Pool
import numpy as np
import pandas as pd

# 1) Train- und Test-Daten vorbereiten
train_fe = train_fe.sort_values("date").reset_index(drop=True)
test_fe = test_fe.sort_values("date").reset_index(drop=True)

# Gemeinsame Typ-Normalisierung für train_fe und test_fe
for df in [train_fe, test_fe]:
    for c in ["family", "family_group"]:
        if c in df.columns:
            df[c] = df[c].astype("category")
    
    if "is_transferred_flag" in df.columns:
        df["is_transferred_flag"] = (
            df["is_transferred_flag"]
            .replace({"True": 1, "False": 0})
            .astype(float)
        )


X_train_boost, y_train_raw, y_train_log = prepare_xy(train_fe, feature_cols, target_col="sales")
X_test_boost, _, _ = prepare_xy(test_fe.assign(sales=0.0), feature_cols, target_col="sales")  # Dummy

# Für CatBoost brauchen wir die rohen Frames (mit category dtypes)
X_train_cat = train_fe[feature_cols].copy()
X_test_cat = test_fe[feature_cols].copy()
cat_features = [c for c in ["store_nbr", "family", "family_group"] if c in X_train_cat.columns]
cat_idx = [X_train_cat.columns.get_loc(c) for c in cat_features]

train_pool = Pool(X_train_cat, label=y_train_log, cat_features=cat_idx)
test_pool = Pool(X_test_cat, cat_features=cat_idx)

# 2) Modelle instanziieren
model_xgb = build_xgb()
model_lgbm = build_lgbm()
model_cat = build_cat()

# 3) Fit auf vollem Train-Set (log-Target)
model_xgb.fit(X_train_boost, y_train_log)
model_lgbm.fit(X_train_boost, y_train_log)
model_cat.fit(train_pool)  # kein eval_set nötig für Full-Train

# 4) Test-Predictions im LOG-Raum
test_pred_xgb_log = model_xgb.predict(X_test_boost)
test_pred_lgbm_log = model_lgbm.predict(X_test_boost)
test_pred_cat_log = model_cat.predict(test_pool)


In [123]:
test_pred_xgb_log

array([ 0.27226114,  0.40900496,  2.386457  , ..., -0.2981261 ,
        0.24249083, -0.22524586], shape=(28512,), dtype=float32)

In [124]:
test_pred_lgbm_log

array([ 0.38265104,  0.95312081,  3.31371278, ..., -0.08963257,
       -0.07323099, -0.09458225], shape=(28512,))

In [125]:
test_pred_cat_log

array([4.79944361, 4.46158036, 3.99482405, ..., 4.69775536, 4.37766495,
       2.88694689], shape=(28512,))

In [104]:
'''
# Debug
bad_row = 14256
bad_col = 39
X_test_cat.iloc[bad_row, bad_col], X_test_cat.columns[bad_col]
'''

('False', 'is_transferred_flag')

In [126]:
X_test_cat.iloc[:,bad_col][14250:14260]

14250    0.0
14251    0.0
14252    0.0
14253    0.0
14254    0.0
14255    0.0
14256    0.0
14257    0.0
14258    0.0
14259    0.0
Name: is_transferred_flag, dtype: float64

In [127]:
# Feste Gewichte
w_x, w_l, w_c = 0.4, 0.4, 0.2

test_ens_log = (
    w_x * test_pred_xgb_log
    + w_l * test_pred_lgbm_log
    + w_c * test_pred_cat_log
)

# Zurück in den Sales-Raum
test_ens = np.expm1(test_ens_log)
test_ens = np.maximum(test_ens, 0.0)  # Sicherheit, keine negativen Verkäufe


In [128]:
# Submission bauen
submission = pd.DataFrame({
    "id": test_fe["id"].values,      # aus originalem test.csv gemergt
    "sales": test_ens
})

submission.to_csv("submission_ensemble.csv", index=False)
print(submission.head())


        id      sales
0  3000888   2.393472
1  3001598   3.208753
2  3002210  20.737366
3  3001668  66.664468
4  3002209  97.134169


In [129]:
# Submission loggen
import mlflow
from pathlib import Path

with mlflow.start_run(run_name="final_test_ensemble"):
    # Gewichte loggen
    mlflow.log_param("w_xgb", w_x)
    mlflow.log_param("w_lgbm", w_l)
    mlflow.log_param("w_cat", w_c)

    # Modelle loggen (optional, hier als Artefakt)
    Path("models_final").mkdir(exist_ok=True)
    model_xgb.save_model("models_final/xgb_full.json")
    model_lgbm.booster_.save_model("models_final/lgbm_full.txt")
    model_cat.save_model("models_final/cat_full.cbm")

    mlflow.log_artifact("models_final/xgb_full.json")
    mlflow.log_artifact("models_final/lgbm_full.txt")
    mlflow.log_artifact("models_final/cat_full.cbm")

    # Submission loggen
    mlflow.log_artifact("submission_ensemble.csv")


🏃 View run final_test_ensemble at: http://127.0.0.1:5000/#/experiments/1/runs/0afb5fc65041467494835db40dc3f110
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
